# Benford's Law Explorer

Interactive testing sandbox for Benford's Law analysis against different cleaned datasets.

## Usage
1. Load a CSV dataset (clean, dirty, or fraud-seeded)
2. Select the amount column
3. Run Benford analysis: first digit, second digit, chi-square, MAD
4. Visualise distributions side-by-side
5. Interpret conformity levels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add src to path so we can import our modules
sys.path.insert(0, str(Path.cwd().parent))

from src.ingestion import load_file, normalise_columns, parse_amount_column, validate_schema
from src.benford import (
    expected_benford_distribution,
    extract_first_digit,
    first_digit_distribution,
    expected_second_digit_distribution,
    extract_second_digit,
    second_digit_distribution,
)
from src.statistics import chi_square_test, mean_absolute_deviation, z_score_by_digit
from src.visualisation import plot_digit_distribution

print("Imports successful!")

## Step 1: Load Dataset

Load one of the sample datasets generated by `scripts/generate_sample_data.py`

In [ ]:
# TODO: Load your dataset here
# Option 1: CSV file
# df = load_file('data/sample/clean_transactions.csv')
# df = load_file('data/sample/dirty_transactions.csv')
# df = load_file('data/sample/fraud_seeded.csv')

# Option 2: Manual inspection of available files
data_dir = Path('data/sample')
if data_dir.exists():
    print(f"Available sample files in {data_dir}:")
    for f in data_dir.glob('*.csv'):
        print(f"  - {f.name}")
else:
    print(f"Run 'python scripts/generate_sample_data.py' first to create sample datasets")

In [ ]:
# Load a dataset (uncomment one)
# df = load_file('data/sample/clean_transactions.csv')
df = load_file('data/sample/fraud_seeded.csv')

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

## Step 2: Normalise and Parse Data

In [ ]:
# Normalise column names
df = normalise_columns(df)
print(f"Normalised columns: {list(df.columns)}")

# Select the amount column (adjust to your dataset)
amount_column = 'amount'  # or 'invoice_amount', depending on your file

# Parse amounts (strip $, handle negatives)
df_clean, df_errors = parse_amount_column(df, amount_column)

print(f"\nParsed {len(df_clean)} valid amounts, {len(df_errors)} errors")
if len(df_errors) > 0:
    print("\nError rows:")
    print(df_errors.head())

# Use only valid amounts
df = df_clean
amounts = df[amount_column]
print(f"\nAmount statistics:")
print(amounts.describe())

## Step 3: First Digit Analysis

In [ ]:
# Compute first-digit distribution
first_dist = first_digit_distribution(amounts)
print(first_dist)

# Run statistical tests
chi_sq = chi_square_test(first_dist['observed_freq'], first_dist['expected_freq'], len(amounts))
mad = mean_absolute_deviation(first_dist['observed_freq'], first_dist['expected_freq'])
z_scores = z_score_by_digit(first_dist['observed_freq'], first_dist['expected_freq'], len(amounts))

print(f"\nChi-Square Test:")
print(f"  Statistic: {chi_sq['statistic']:.4f}")
print(f"  P-value: {chi_sq['p_value']:.4f}")
print(f"  Significant: {chi_sq['significant']}")
print(f"  Interpretation: {chi_sq['interpretation']}")

print(f"\nMean Absolute Deviation:")
print(f"  MAD: {mad['mad']:.6f}")
print(f"  Conformity Level: {mad['conformity_level']}")
print(f"  Flagged: {mad['flagged']}")

print(f"\nZ-Scores by Digit (most anomalous first):")
print(z_scores)

## Step 4: Visualise First Digit Distribution

In [ ]:
# Create side-by-side plot
fig, ax = plt.subplots(figsize=(10, 5))
plot_digit_distribution(
    first_dist,
    digit_type='first',
    title=f'First Digit Distribution (n={len(amounts)})',
    ax=ax
)
plt.tight_layout()
plt.show()

## Step 5: Second Digit Analysis

In [ ]:
# Compute second-digit distribution
second_dist = second_digit_distribution(amounts)
print(second_dist)

# Run statistical tests
chi_sq_2 = chi_square_test(second_dist['observed_freq'], second_dist['expected_freq'], len(amounts))
mad_2 = mean_absolute_deviation(second_dist['observed_freq'], second_dist['expected_freq'])
z_scores_2 = z_score_by_digit(second_dist['observed_freq'], second_dist['expected_freq'], len(amounts))

print(f"\nMean Absolute Deviation (Second Digit):")
print(f"  MAD: {mad_2['mad']:.6f}")
print(f"  Conformity Level: {mad_2['conformity_level']}")
print(f"  Flagged: {mad_2['flagged']}")

## Step 6: Visualise Second Digit Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_digit_distribution(
    second_dist,
    digit_type='second',
    title=f'Second Digit Distribution (n={len(amounts)})',
    ax=ax
)
plt.tight_layout()
plt.show()

## Step 7: Interpretation & Next Steps

### Conformity Levels (MAD Thresholds)
- **< 0.006**: Close conformity (data appears genuine)
- **< 0.012**: Acceptable conformity (minor deviations acceptable)
- **< 0.015**: Marginally acceptable (investigate further)
- **>= 0.015**: Non-conformity (likely fraud or data quality issues)

### What to look for:
1. **First digit flagged**: Possible systematic fraud (e.g., threshold avoidance)
2. **Second digit flagged**: Less common, but still suspicious
3. **Specific digits with high Z-scores**: Filter transactions starting/containing that digit
4. **Chi-square significance**: If p < 0.05, reject null hypothesis (distribution ≠ Benford)

### Next:
- Run `main.py` on the full dataset with duplicate detection, round-number detection
- Use `dashboard.py` for interactive exploration of vendor-level risk scores
- See Phase 12 (Integration Tests) for end-to-end validation

In [ ]:
# Summary table: compare both digits
summary = pd.DataFrame([
    {'Test': 'First Digit MAD', 'Value': f"{mad['mad']:.6f}", 'Conformity': mad['conformity_level']},
    {'Test': 'Second Digit MAD', 'Value': f"{mad_2['mad']:.6f}", 'Conformity': mad_2['conformity_level']},
    {'Test': 'First Digit Chi-Sq', 'Value': f"{chi_sq['p_value']:.4f}", 'Significant': chi_sq['significant']},
    {'Test': 'Second Digit Chi-Sq', 'Value': f"{chi_sq_2['p_value']:.4f}", 'Significant': chi_sq_2['significant']},
])
print("\n=== SUMMARY ===")
print(summary.to_string(index=False))